# 3. Model Analysis

This notebook analyzes model predictions and investigates individual creatures.

**Input:**
- `helper_files/engineered_features.parquet`
- `pickled_models/hp_model_cr*.pkl`

**Output:**
- `data/engineered_features.csv`
- `data/feature_contributions.csv`

## Imports and Configs

In [9]:
import numpy as np
import os
import pandas as pd
import sys

from pathlib import Path

In [10]:
# Detect execution context and set paths dynamically
sys.path.insert(0, '.')

# Get the current working directory
cwd = Path.cwd()

# Check if we're in the notebooks directory or project root
if cwd.name == 'notebooks':
    # Running from notebooks directory (in Jupyter)
    DATA_DIR = '../data'
    PICKLED_MODELS_DIR = '../pickled_models'
    MONSTER_BUILDER_DIR = '../monster-builder-v2'
    HELPERS_DIR = './helper_files'
    IN_NB_DIR = False
else:
    # Running from project root (via run_three_tier_model.py)
    DATA_DIR = './data'
    PICKLED_MODELS_DIR = './pickled_models'
    MONSTER_BUILDER_DIR = './monster-builder-v2'
    HELPERS_DIR = './notebooks/helper_files'
    IN_NB_DIR = False

print(f"📁 Execution context detected:")
print(f"   Current directory: {cwd}")
print(f"   Data directory: {DATA_DIR}")
print(f"   Models directory: {PICKLED_MODELS_DIR}")

print("Imports successful")

📁 Execution context detected:
   Current directory: /workspaces/matrix_v0
   Data directory: ./data
   Models directory: ./pickled_models
Imports successful


In [11]:
# Add helper_files to path
if IN_NB_DIR is True:
    from helper_files import (
        get_phase3_features,
        get_cr_tier,
        PHASE2_PENALTIES,
        load_model,
        add_percentile_by_cr,
        investigate_creature,
    )

    print("Imports successful")
else:
    from notebooks.helper_files import (
        get_phase3_features,
        get_cr_tier,
        PHASE2_PENALTIES,
        load_model,
        add_percentile_by_cr,
        investigate_creature,
    )

    print("Imports successful")


Imports successful


## Load Data and Models

In [12]:
# Load engineered features
load_path = HELPERS_DIR + "/engineered_features.parquet"
df = pd.read_parquet(load_path)
print(f"Loaded {len(df)} monsters")

Loaded 382 monsters


In [13]:
# Load models
models = {}
scalers = {}
os.makedirs(PICKLED_MODELS_DIR, exist_ok=True)
load_path = PICKLED_MODELS_DIR + "/hp_model_tier.pkl"
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    filepath = load_path.replace('tier', tier)
    data = load_model(filepath)
    models[tier] = data['model']
    scalers[tier] = data['scaler']
    print(f"Loaded {tier} model")

phase3_features = get_phase3_features()

Loaded cr1 model
Loaded cr2 model
Loaded cr3 model
Loaded cr4 model
Loaded cr5 model


## Generate Predictions

In [14]:
def get_prediction_for_creature(row):
    """Get the final HP prediction for a creature based on its CR tier."""
    tier = row['cr_tier']
    
    # Get features
    X = row[phase3_features].to_frame().T.fillna(0).infer_objects(copy=False).values.reshape(1, -1)
    
    # Scale and predict
    X_scaled = scalers[tier].transform(X)
    residual_pred = models[tier].predict(X_scaled)[0]
    
    return row['hp_after_phase2'] + residual_pred

df['predicted_hp'] = df.apply(get_prediction_for_creature, axis=1)
df['hp_delta'] = df['predicted_hp'] - df['actual_hp']
df['hp_delta_pct'] = (df['hp_delta'] / df['actual_hp']) * 100

df = add_percentile_by_cr(df)

print("Predictions generated")

/tmp/ipykernel_58914/464985233.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = row[phase3_features].to_frame().T.fillna(0).infer_objects(copy=False).values.reshape(1, -1)
/tmp/ipykernel_58914/464985233.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = row[phase3_features].to_frame().T.fillna(0).infer_objects(copy=False).values.reshape(1, -1)
/tmp/ipykernel_58914/464985233.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) in

Predictions generated


/tmp/ipykernel_58914/464985233.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = row[phase3_features].to_frame().T.fillna(0).infer_objects(copy=False).values.reshape(1, -1)
/tmp/ipykernel_58914/464985233.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = row[phase3_features].to_frame().T.fillna(0).infer_objects(copy=False).values.reshape(1, -1)
/tmp/ipykernel_58914/464985233.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) in

In [15]:
# Summary statistics
print("Overall Prediction Summary:")
print(f"   Mean HP Error: {df['hp_delta'].mean():.1f} HP")
print(f"   Mean Absolute Error: {df['hp_delta'].abs().mean():.1f} HP")
print(f"   Mean % Error: {df['hp_delta_pct'].mean():.1f}%")
print(f"   Mean Absolute % Error: {df['hp_delta_pct'].abs().mean():.1f}%")

Overall Prediction Summary:
   Mean HP Error: 29.2 HP
   Mean Absolute Error: 35.4 HP
   Mean % Error: 242.3%
   Mean Absolute % Error: 256.6%


In [16]:
# Summary by CR tier
tier_labels = {
    'cr1': 'CR < 1',
    'cr2': 'CR 1-4',
    'cr3': 'CR 5-10',
    'cr4': 'CR 11-16',
    'cr5': 'CR > 16',
}

print("\nBy CR Tier:")
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    tier_df = df[df['cr_tier'] == tier]
    mae = tier_df['hp_delta'].abs().mean()
    mape = tier_df['hp_delta_pct'].abs().mean()
    print(f"   {tier_labels[tier]:12s}: MAE={mae:6.1f} HP, MAPE={mape:5.1f}%")


By CR Tier:
   CR < 1      : MAE=  28.3 HP, MAPE=574.6%
   CR 1-4      : MAE=  39.7 HP, MAPE=104.5%
   CR 5-10     : MAE=  38.4 HP, MAPE= 37.3%
   CR 11-16    : MAE=  49.4 HP, MAPE= 28.6%
   CR > 16     : MAE=  29.8 HP, MAPE=  8.8%


## Feature Contributions

In [17]:
def calculate_feature_contributions(row):
    """Calculate HP contribution of each feature for a creature."""
    tier = row['cr_tier']
    penalties = PHASE2_PENALTIES[tier]
    
    contributions = {
        'Name': row['Name'],
        'CR': row['cr_numeric'],
        'actual_hp': row['actual_hp'],
        'predicted_hp': row['predicted_hp'],
        'hp_error': row['hp_delta'],
        'hp_error_pct': row['hp_delta_pct'],
        'hp_baseline': row['hp_baseline'],
        'hp_after_phase1_5': row['hp_after_phase1_5'],
        'hp_after_phase2': row['hp_after_phase2'],
        'phase1_5_resistance_penalty': row.get('resistance_penalty', 0),
        'phase1_5_immunity_penalty': row.get('immunity_penalty', 0),
        'phase1_5_total_penalty': row.get('total_defensive_penalty', 0),
    }
    
    # Phase 2 contributions
    contributions['phase2_ac_contribution'] = row['ac_deviation'] * penalties.get('ac_deviation', 0)
    contributions['phase2_attack_contribution'] = row['attack_deviation'] * penalties.get('attack_deviation', 0)
    contributions['phase2_dpr_contribution'] = row['dpr_deviation'] * penalties.get('dpr_deviation', 0)
    contributions['phase2_save_dc_contribution'] = row['save_dc_deviation'] * penalties.get('save_dc_deviation', 0)
    contributions['phase2_flying_contribution'] = row['has_flying'] * penalties.get('has_flying', 0)
    contributions['phase2_advantage_contribution'] = row.get('has_advantage_condition', 0) * penalties.get('has_advantage_condition', 0)
    contributions['phase2_disadvantage_contribution'] = row.get('has_disadvantage_condition', 0) * penalties.get('has_disadvantage_condition', 0)
    contributions['phase2_attackers_advantage_contribution'] = row.get('has_attackers_advantage', 0) * penalties.get('has_attackers_advantage', 0)
    contributions['phase2_prone_contribution'] = row.get('inflicts_prone', 0) * penalties.get('inflicts_prone', 0)
    
    contributions['phase2_total_contribution'] = sum([
        contributions['phase2_ac_contribution'],
        contributions['phase2_attack_contribution'],
        contributions['phase2_dpr_contribution'],
        contributions['phase2_save_dc_contribution'],
        contributions['phase2_flying_contribution'],
        contributions['phase2_advantage_contribution'],
        contributions['phase2_disadvantage_contribution'],
        contributions['phase2_attackers_advantage_contribution'],
        contributions['phase2_prone_contribution'],
    ])
    
    # Phase 3 contributions
    X = np.array([[row.get(f, 0) for f in phase3_features]])
    X = np.nan_to_num(X, 0)
    X_scaled = scalers[tier].transform(X)[0]
    coefs = models[tier].coef_
    
    phase3_total = 0
    for i, feature in enumerate(phase3_features):
        contrib = X_scaled[i] * coefs[i]
        contributions[f'phase3_{feature}'] = contrib
        phase3_total += contrib
    
    contributions['phase3_intercept'] = models[tier].intercept_
    contributions['phase3_total_contribution'] = phase3_total + contributions['phase3_intercept']
    
    return pd.Series(contributions)

# Calculate contributions for all creatures
contributions_df = df.apply(calculate_feature_contributions, axis=1)
print(f"Calculated contributions for {len(contributions_df)} creatures")

Calculated contributions for 382 creatures


## Export Data

In [19]:
# Export engineered features
export_columns = [
    'Name', 'Type', 'Size', 'Challenge_Rating', 'cr_numeric', 'cr_tier',
    'HP', 'actual_hp', 'AC', 'ac_value',
    'predicted_hp', 'hp_delta', 'hp_delta_pct', 'hp_delta_pct_percentile',
    'hp_baseline', 'ac_baseline', 'attack_baseline', 'dpr_baseline', 'dc_baseline',
    'highest_attack_bonus', 'highest_save_dc', 'estimated_dpr', 'legendary_dpr', 'total_dpr',
    'ac_deviation', 'attack_deviation', 'dpr_deviation', 'save_dc_deviation',
    'hp_after_phase1_5', 'hp_after_phase2', 'residual_hp',
]

# Add all existing columns that match
export_cols = [c for c in export_columns if c in df.columns]

export_df = df[export_cols].sort_values('cr_numeric')

export_path = DATA_DIR + '/engineered_features.csv'
export_df.to_csv(export_path, index=False)
print(f"Exported {len(export_df)} monsters to data/engineered_features.csv")

Exported 382 monsters to data/engineered_features.csv


In [20]:
# Export contributions
export_path = DATA_DIR + '/feature_contributions.csv'
contributions_df.to_csv(export_path, index=False)
print(f"Exported contributions to data/feature_contributions.csv")

Exported contributions to data/feature_contributions.csv


## Investigate Specific Creatures

In [21]:
# Example: Investigate a creature
investigate_creature('Elephant', df, contributions_df)

  ELEPHANT (CR 4.0)

Actual HP:              76
Predicted HP:          120
Error:                  44  (  58.3%)

--------------------------------------------------------------------------------

PHASE 1: CR BASELINE
  HP Baseline (CR 4.0):                               101

PHASE 1.5: RESISTANCES & IMMUNITIES
  No resistances or immunities
  HP after Phase 1.5:                                   101

PHASE 2: COMBAT STATS
  AC Contribution                                      5
  Attack Bonus Contribution                           -6
  DPR Contribution                                    -6
  Save DC Contribution                                10
  Inflicts Prone                                      -1
  -----------------------------------------------------
  Phase 2 Total:                                          2
  HP after Phase 2:                                     103

PHASE 3: INDIVIDUAL FEATURES
  Feature: has_legendary_resistance_scaled     value:    0.0  hp impact:     -0
  F

In [22]:
# List worst predictions
print("\nWorst Over-predictions (predicted > actual):")
over_pred = df[df['hp_delta'] > 0].nlargest(10, 'hp_delta_pct')[['Name', 'cr_numeric', 'actual_hp', 'predicted_hp', 'hp_delta_pct']]
print(over_pred.to_string(index=False))


Worst Over-predictions (predicted > actual):
           Name  cr_numeric  actual_hp  predicted_hp  hp_delta_pct
Avatar of Death         0.0          1     63.613939   6261.393933
           Frog         0.0          1     55.064641   5406.464069
            Rat         0.0          1     44.897091   4389.709051
       Seahorse         0.0          1     36.578813   3557.881265
            Owl         0.0          1     35.218705   3421.870537
            Bat         0.0          1     34.585187   3358.518698
           Hawk         0.0          1     34.053174   3305.317411
      Sea Horse         0.0          1     31.901186   3090.118576
        Piranha         0.0          1     26.533163   2553.316287
         Weasel         0.0          1     24.397296   2339.729641


In [23]:
print("\nWorst Under-predictions (predicted < actual):")
under_pred = df[df['hp_delta'] < 0].nsmallest(10, 'hp_delta_pct')[['Name', 'cr_numeric', 'actual_hp', 'predicted_hp', 'hp_delta_pct']]
print(under_pred.to_string(index=False))


Worst Under-predictions (predicted < actual):
                  Name  cr_numeric  actual_hp  predicted_hp  hp_delta_pct
              Scorpion        0.00          1     -8.593843   -959.384324
Copper Dragon Wyrmling        1.00         22    -19.953683   -190.698557
                Sprite        0.25          2     -1.122030   -156.101499
                Dretch        0.25         18     -6.476532   -135.980733
Silver Dragon Wyrmling        2.00         45      1.366769    -96.962736
 Giant Poisonous Snake        0.25         11      1.452161    -86.798538
                Couatl        4.00         97     13.262732    -86.327081
       Giant Centipede        0.25          4      1.508807    -62.279815
                  Drow        0.25         13      5.439044    -58.161198
Animated Object (Huge)        0.00         40     17.099225    -57.251938


In [24]:
print("\nBest Predictions:")
best = df.nsmallest(10, df['hp_delta_pct'].abs())[['Name', 'cr_numeric', 'actual_hp', 'predicted_hp', 'hp_delta_pct']]
print(best.to_string(index=False))


Best Predictions:


KeyError: 0.16710733417760082

In [25]:
# Interactive investigation
# Uncomment and modify to investigate specific creatures:
# investigate_creature('Adult Red Dragon', df, contributions_df)
# investigate_creature('Tarrasque', df, contributions_df)
investigate_creature('Goblin', df, contributions_df)

  GOBLIN (CR 0.25)

Actual HP:               7
Predicted HP:           46
Error:                  39  ( 562.4%)

--------------------------------------------------------------------------------

PHASE 1: CR BASELINE
  HP Baseline (CR 0.25):                                20

PHASE 1.5: RESISTANCES & IMMUNITIES
  No resistances or immunities
  HP after Phase 1.5:                                    20

PHASE 2: COMBAT STATS
  AC Contribution                                     -6
  Attack Bonus Contribution                           -2
  DPR Contribution                                    -0
  Save DC Contribution                                38
  -----------------------------------------------------
  Phase 2 Total:                                         30
  HP after Phase 2:                                      50

PHASE 3: INDIVIDUAL FEATURES
  Feature: has_legendary_resistance_scaled     value:    0.0  hp impact:      0
  Feature: has_magic_resistance_scaled         value:    0.0